QUESTION 2
1. Write ADQL query for Gaia DR3.
2. Identify bad 2MASS photometry and non-positive Gaia parallax.
3. Produce 2-panel figure of Gaia BP-RP vs absG, and 2MASS J-Ks vs appH.
4. Observing Proposal Recommendation

In [2]:
#PART 1 --- ADQL Query 
from astroquery.gaia import Gaia
import os

#Defining the query --- what I'm specifically grabbing and from where.
query = """
SELECT
    gaia.source_id, gaia.ra, gaia.dec, gaia.parallax, gaia.pmra, gaia.pmdec, gaia.phot_g_mean_mag,
    gaia.bp_rp, tmass.designation AS tmass_designation, tmass.j_m, tmass.h_m, tmass.ks_m, tmass.ph_qual
FROM gaiadr3.gaia_source AS gaia
JOIN gaiadr3.tmass_psc_xsc_best_neighbour AS xmatch
    ON gaia.source_id = xmatch.source_id
JOIN gaiadr3.tmass_psc_xsc_join AS xjoin
    ON xmatch.clean_tmass_psc_xsc_oid = xjoin.clean_tmass_psc_xsc_oid
JOIN gaiadr1.tmass_original_valid AS tmass
    ON xjoin.original_psc_source_id = tmass.designation
WHERE 1 = CONTAINS(POINT('ICRS', gaia.ra, gaia.dec), CIRCLE('ICRS', 132.825, 11.8, 1.0))
    AND gaia.phot_g_mean_mag < 14
"""
#Lodge query (asynchronously, to avoid timeouts etc.)
job = Gaia.launch_job_async(query, dump_to_file=False)
M67data = job.get_results()

#Report the number of stars retrieved from the query.
print(f"Number of stars: {len(M67data)}")

#Write the results to a CSV file in the "data" directory, creating the directory if it doesn't exist.
os.makedirs("data", exist_ok=True)
M67data.write("data/M67data.csv", format="ascii.csv", overwrite=True)



INFO: Query finished. [astroquery.utils.tap.core]
Number of stars: 1018


In [ ]:
#Part 2 --- Bad Data
import numpy as np

#Create mask for bad photometry.
bad_photometry_mask = M67data['ph_qual'] != 'AAA'
n_bad_photometry = bad_photometry_mask.sum()
print(f"Number of stars with bad photometry: {n_bad_photometry}")

#Create mask for bad parallax (parallax <= 0 or NaN).
bad_parallax_mask = (M67data['parallax'] <= 0) | np.isnan(M67data['parallax'])
n_bad_parallax = bad_parallax_mask.sum()
print(f"Number of stars with bad parallax: {n_bad_parallax}")

#Filter the data and write to a new CSV file.
filtered_data = M67data[~bad_photometry_mask & ~bad_parallax_mask]
print(f"Number of stars with good data: {len(filtered_data)}")
filtered_data.write("data/M67data_filtered.csv", format="ascii.csv", overwrite=True)


Number of stars with bad photometry: 21
Number of stars with bad parallax: 2
Number of stars with good data: 988
